# DMRC Contract Intelligence — Retrieval Validation
## Notebook 03 · Complete Retrieval-Pipeline Validation

**Project:** Enterprise RAG platform for DMRC contract intelligence (`dmrc_deploy`).
**Scope:** end-to-end validation of every retrieval capability the `/ask` API depends on — Clause Direct, Clause Hybrid, BOQ Direct, BOQ Hybrid, metadata integrity, exact page / page-image resolution, and related-page retrieval — followed by Direct-vs-Hybrid comparison, quantitative metrics, and failure analysis.

**Companion notebooks:** `01_Setup_and_Retrieval_Validation.ipynb` (original retrieval smoke-tests) and `02_Gemma_Inference_and_Serving.ipynb` (LLM stage). This notebook is **CPU-only** and does not load Gemma.

**Repository audit summary** (verified against the current `dmrc_deploy` source before this notebook was written):

| Capability | Status | Where |
|---|---|---|
| Clause Direct Retrieval | ✅ Implemented | `query.extract_clause_no` → `get_chunk_by_clause_no` + `get_chunks_by_parent_clause` (app.py fast path) |
| Clause Hybrid Retrieval | ✅ Implemented | `hybrid_retriever.hybrid_search` (dense + BM25 + normalized fusion) → `reranker.rerank` → `expand_with_siblings` |
| BOQ Direct Retrieval | ✅ Implemented | `query.extract_boq_item_no` → `get_chunk_by_boq_item_no` (matches `parent`/`s_no`/`item_header_no`/`section_no`) |
| BOQ Hybrid Retrieval | ✅ Shared pipeline | same collection + BM25 index; BOQ-only mode via `metadata_filter={"chunk_type": "boq"}` (supported by dense, BM25 and hybrid paths) |
| Exact PDF page metadata | ✅ Implemented | `pdf_page` on **all 353 chunks**; citation stamp via `printed_page`/`stamp_number` (Rule 1 label vs Rule 2 file key) |
| Exact PDF page images | ✅ clause / ❌ BOQ | `scripts/render_pages.py` → `page_images/{document_id}/pNNNN.jpg`, served at `/pages`. **Verified against the shipped `chroma_db/`: all 63 clause chunks resolve an image; 0 of 290 BOQ chunks do** — their `document_id` values (`BOQ-CONTRACT-AGREEMENT-CE-10-CE-11-…`, `BOQ-CE-10-AND-11-LOT-4-SCHEDULE-S4-1`) don't match any `page_images/` directory (`…CE-10-AND-11…`). See §11 and §15. |
| Related page **images** | ⚠️ Partial | related *clauses* via `expand_with_siblings` (their pages come along); embedded-figure code exists but `figure_images/manifest.json` is **not committed**; **no neighboring-page (±1) helper exists** — one is defined and validated in Section 12 |
| Metadata preservation | ✅ Implemented | per-type schema (`clause_no`, `parent_clause`, `document_name`, `pdf_page`, … / `s_no`, `section_no`, `unit`, `quantities`, `page_label`, …) |
| Metadata filtering | ✅ Implemented | `where=` filter threaded through `query.search`, `BM25Index.search`, `hybrid_search` |
| Separate Clause/BOQ indexes | ❌ Single collection | one ChromaDB collection `dmrc_be12be14_ecs` + one BM25 index, discriminated by `chunk_type` |
| Embedding / VDB / reranker | — | `BAAI/bge-m3` (1024-d) / ChromaDB (persistent) / `BAAI/bge-reranker-v2-m3`; fusion = min-max score normalization + max-merge (not RRF) |


## 1. Environment Setup

**Purpose** — Install the exact pinned dependency versions the retrieval stack was validated against, minus the GPU-only 4-bit extras.

**Explanation** — Identical to notebook 01: `bitsandbytes` / `nvidia-nvjitlink-cu13` are stripped so this notebook runs on a plain **CPU** runtime. Skip this cell if you already ran notebook 01 in this session.

In [ ]:
import os
REQ = next((p for p in ("/content/dmrc_deploy/requirements.txt", "dmrc_deploy/requirements.txt",
                        "requirements.txt") if os.path.isfile(p)), None)
assert REQ, "requirements.txt not found -- run Section 2 (Repository Setup) first, then re-run this cell."
!grep -v -E "^(bitsandbytes|nvidia-nvjitlink-cu13)" {REQ} > /tmp/requirements_retrieval.txt
%pip install -q -r /tmp/requirements_retrieval.txt
print("Dependencies installed. If a numpy import error appears later, restart the runtime once and re-run from Section 2.")

## 2. Repository Setup

**Purpose** — Obtain a current working copy of `dmrc_deploy`, including the pre-built `chroma_db/` collection and rendered `page_images/`.

**Explanation** — Idempotent clone-or-pull, matching notebooks 01/02.

In [ ]:
import os
os.chdir("/content" if os.path.isdir("/content") else os.getcwd())
!test -d dmrc_deploy && (echo "dmrc_deploy/ already present -- pulling latest" && cd dmrc_deploy && git pull) \
    || git clone https://github.com/sunvantaconsultancysolutions-design/dmrc_deploy
os.chdir("dmrc_deploy")
!ls -d src chroma_db page_images data

### Shared helpers

**Purpose** — Load the validation helpers used by every section below. All retrieval logic is imported from the repository's own `src/` modules; the helpers only add lazy model loading, page-image resolution (mirroring `app.py`'s Rule 1/Rule 2), display tables, and a `neighbor_page_images()` related-page helper (Section 12).

Every later section calls `ensure_models()` itself, so after Sections 1–2 have run once, **each section is independently executable**.

In [ ]:

# ============================================================================
# Shared retrieval-validation helpers (self-contained; safe to re-run).
#
# Everything here is a thin display/validation layer over the repository's
# own modules -- no retrieval logic is duplicated. Lazy loading means any
# section of this notebook can be run independently after Sections 1-2
# (clone + install) have been executed once in the session.
# ============================================================================
import os, sys, re, time, json

# Resolve the repository root the same way for Colab (/content) and local runs.
for _cand in ("/content/dmrc_deploy", os.path.abspath("dmrc_deploy"), os.path.abspath(".")):
    if os.path.isfile(os.path.join(_cand, "src", "query.py")):
        REPO = _cand
        break
else:
    raise RuntimeError("dmrc_deploy repository not found -- run the Repository Setup section first.")
os.chdir(REPO)                      # storage.py resolves CHROMA_PATH against CWD
if REPO not in sys.path:
    sys.path.insert(0, REPO)

from IPython.display import display, Markdown, Image as IPyImage

# --- Repository modules (imported, never duplicated) ------------------------
from src import query as q                       # dense search + exact fast paths
from src.hybrid_retriever import hybrid_search   # dense + BM25 + normalized fusion
from src.bm25_index import get_bm25_index        # in-memory BM25 over the corpus
from src.reranker import rerank, evaluate_confidence, expand_with_siblings
from src.prompt_engineering import (
    get_document_name, get_scanned_page, get_boq_item_number, get_boq_page_number,
)
from src.storage import get_collection, COLLECTION_NAME

# NOTE: src.retrieval_caps is deliberately NOT imported in this notebook.
# It monkey-patches hybrid_search()/rerank() in place (as app.py needs in
# production); for validation we want the *uncapped* functions so full
# candidate lists are observable. app.py's behaviour is still exercised,
# because the caps only truncate list length, never change ranking.

_MODELS_READY = False
def ensure_models():
    """Idempotently load the dense encoder, BM25 index, and reranker."""
    global _MODELS_READY
    if _MODELS_READY:
        return
    t0 = time.time()
    q.get_model()                                   # BAAI/bge-m3 (dense)
    get_bm25_index()                                # builds BM25 over ChromaDB corpus
    from src.reranker import get_reranker_model
    get_reranker_model()                            # BAAI/bge-reranker-v2-m3
    _MODELS_READY = True
    print(f"Models + indexes ready in {time.time()-t0:.1f}s "
          f"(collection: {COLLECTION_NAME}, {get_collection().count()} vectors)")

# --- Page-image resolution (mirrors app.py's Rule 1 / Rule 2 exactly) -------
PAGE_IMAGES_DIR = os.path.join(REPO, "page_images")

def resolve_pdf_page(md):
    """Rule 2: the FILE-LOOKUP key is pdf_page (page_number fallback for BOQ)."""
    p = md.get("pdf_page")
    if p in (None, "") and md.get("chunk_type") == "boq":
        p = md.get("page_number")
    try:
        return int(p)
    except (TypeError, ValueError):
        return None

def page_image_path(md):
    doc_id, p = md.get("document_id"), resolve_pdf_page(md)
    if not doc_id or p is None:
        return None
    path = os.path.join(PAGE_IMAGES_DIR, doc_id, f"p{p:04d}.jpg")
    return path if os.path.isfile(path) else None

def show_page_image(md, caption="", width=520):
    path = page_image_path(md)
    if path:
        display(Markdown(f"**{caption}** &nbsp; `{os.path.relpath(path, REPO)}`"))
        display(IPyImage(filename=path, width=width))
        return True
    display(Markdown(f"**{caption}** — *no rendered page image for "
                     f"(document_id={md.get('document_id')!r}, pdf_page={resolve_pdf_page(md)!r})*"))
    return False

def neighbor_page_images(md, radius=1):
    """Related-page helper: returns existing image paths for pdf_page +/- radius
    within the same document_id, using the same AVAILABLE_PAGES logic as app.py.
    As of the Task 2 fix this is now also shipped in SourceItem as
    prev_image_url / next_image_url; this local helper is retained here for
    independent notebook validation without starting the FastAPI server."""
    doc_id, p = md.get("document_id"), resolve_pdf_page(md)
    if not doc_id or p is None:
        return []
    out = []
    for delta in range(-radius, radius + 1):
        if delta == 0:
            continue
        path = os.path.join(PAGE_IMAGES_DIR, doc_id, f"p{p+delta:04d}.jpg")
        if os.path.isfile(path):
            out.append((p + delta, path))
    return out

# --- Uniform result accessors ------------------------------------------------
def meta(hit):      return hit.get("metadata") or {}
def clause_no(hit): return meta(hit).get("clause_no") or "-"
def boq_item(hit):  return get_boq_item_number(meta(hit)) or "-"
def doc_name(hit):  return get_document_name(meta(hit)) or meta(hit).get("document_id") or "-"
def scan_page(hit):
    s = get_scanned_page(meta(hit))
    return "-" if s in (None, "") else str(s)
def score_of(hit):
    for k in ("reranker_score", "score", "similarity_score", "bm25_score"):
        if hit.get(k) is not None:
            return round(float(hit[k]), 4)
    return None
def preview(hit, n=170):
    t = (hit.get("document") or "").replace("\n", " ")
    return t[: n] + ("…" if len(t) > n else "")

def hits_table(hits, kind="clause", top=5, title=None):
    """Render a compact markdown table for a hit list."""
    rows = []
    for h in hits[:top]:
        ident = clause_no(h) if kind == "clause" else boq_item(h)
        rows.append(f"| {ident} | {preview(h, 88)} | {doc_name(h)[:38]} | "
                    f"{scan_page(h)} | {resolve_pdf_page(meta(h))} | "
                    f"{h.get('retrieval_source','-')} | {score_of(h)} |")
    head = (f"| {'Clause' if kind=='clause' else 'BOQ item'} | Text | Document | "
            "Stamped pg | PDF pg | Source | Score |\n|---|---|---|---|---|---|---|")
    display(Markdown(((f"**{title}**\n\n") if title else "") + head + "\n" + "\n".join(rows)))

# --- Retrieval wrappers used throughout both notebooks -----------------------
def clause_direct(query, top_k=5):
    """Direct clause retrieval = app.py's fast path when the query names a
    clause number (exact metadata lookup + parent-family expansion),
    otherwise clause-filtered pure dense search. The exact path is
    metadata-only -- no model load needed, exactly as in production."""
    no = q.extract_clause_no(query)
    if no:
        hits = list(q.get_chunk_by_clause_no(no))
        seen = {h["chunk_id"] for h in hits}
        for child in q.get_chunks_by_parent_clause(no):
            if child["chunk_id"] not in seen:
                hits.append(dict(child, score=1.0,
                                 retrieval_source="exact_clause_family_match"))
        if hits:
            return hits, f"exact_clause_match ({no})"
    ensure_models()
    hits = q.search(query, top_k=top_k, metadata_filter={"chunk_type": "clause"})
    for h in hits:
        h.setdefault("retrieval_source", "dense")
    return hits, "dense (clause-filtered)"

def boq_direct(query, top_k=5):
    """Direct BOQ retrieval = app.py's BOQ fast path (exact identifier
    lookup on parent/s_no/item_header_no/section_no), otherwise
    BOQ-filtered pure dense search."""
    no = q.extract_boq_item_no(query)
    if no:
        hits = q.get_chunk_by_boq_item_no(no)
        if hits:
            return hits, f"exact_boq_item_match ({no})"
    ensure_models()
    hits = q.search(query, top_k=top_k, metadata_filter={"chunk_type": "boq"})
    for h in hits:
        h.setdefault("retrieval_source", "dense")
    return hits, "dense (boq-filtered)"

def hybrid_pipeline(query, chunk_type=None, top_n=5, expand=False):
    """Full hybrid stack: BM25 + dense -> normalized fusion -> cross-encoder
    rerank (-> optional sibling expansion), with per-stage outputs returned
    so each stage can be displayed and compared."""
    ensure_models()
    flt = {"chunk_type": chunk_type} if chunk_type else None
    bm25 = get_bm25_index().search(query, top_k=10, metadata_filter=flt)
    dense = q.search(query, top_k=10, metadata_filter=flt)
    fused = hybrid_search(query, top_k_dense=30, top_k_bm25=30,
                          final_top_k=60, metadata_filter=flt)
    reranked = rerank(query, fused, top_n=top_n)
    conf = evaluate_confidence(reranked)
    if expand:
        reranked = expand_with_siblings(query, reranked)
    return {"bm25": bm25, "dense": dense, "hybrid": fused,
            "reranked": reranked, "confidence": conf}

print("Helpers loaded. Repository root:", REPO)


## 3. Load Models

**Purpose** — Load the dense encoder (`BAAI/bge-m3`), the in-memory BM25 index, and the cross-encoder reranker (`BAAI/bge-reranker-v2-m3`) once, up front.

**Explanation** — All three are cached at module level inside the repository; `ensure_models()` simply forces the lazy loads now so first-query latency in later sections reflects retrieval, not model download.

In [ ]:
ensure_models()
import torch, chromadb, sentence_transformers
print(f"torch {torch.__version__} | chromadb {chromadb.__version__} | "
      f"sentence_transformers {sentence_transformers.__version__}")
print(f"Dense model : {q.MODEL_NAME}")
from src import reranker as _rr
print(f"Reranker    : {_rr.MODEL_NAME}")

## 4. Build Indexes

**Purpose** — Confirm both indexes the pipeline depends on are intact: the persisted ChromaDB dense index and the BM25 lexical index built over the same corpus.

**Explanation** — The dense index ships pre-built in `chroma_db/` (nothing to rebuild); BM25 is constructed in-memory on first use by `bm25_index.build_bm25_index()`. This section verifies counts by `chunk_type` — the single-collection design means "separate Clause and BOQ indexes" are **logical** (metadata-discriminated), not physical.

In [ ]:
ensure_models()
col = get_collection()
res = col.get(include=["metadatas"])
from collections import Counter
by_type = Counter(m.get("chunk_type") for m in res["metadatas"])
print(f"Collection {COLLECTION_NAME}: {col.count()} vectors -> {dict(by_type)}")

idx = get_bm25_index()
vocab = len(getattr(getattr(idx, "_bm25", None), "idf", {}) or {})
print(f"BM25 index : {len(idx.chunk_ids)} documents, {vocab} vocabulary terms (rank_bm25 BM25Okapi)")
assert col.count() == len(idx.chunk_ids), "BM25 and dense corpora out of sync -- POST /admin/reload-bm25 in production"
print("Dense and BM25 corpora are in sync.")

## 5. Validate Clause Direct Retrieval

**Purpose** — Prove the exact clause-number fast path works for all three shapes present in this corpus: a leaf clause, a parent clause with a self-row + children, and a clause family that exists **only** as children (no self-row).

**Explanation** — `clause_direct()` reproduces `app.py`'s `/ask` fast-path logic verbatim: `extract_clause_no()` → `get_chunk_by_clause_no()` → unconditional `get_chunks_by_parent_clause()` family expansion. Falls back to clause-filtered dense search when no clause number is named.

In [ ]:
for query_text in ["Explain clause 1.2.1",          # leaf clause
                   "Explain Clause 6.8 in detail",   # parent with self-row + 8 children
                   "What does clause 6.7.2 cover?"]: # children-only family (6.7.2-1..-4)
    hits, path = clause_direct(query_text)
    hits_table(hits, "clause", top=10, title=f"“{query_text}” → path: `{path}` ({len(hits)} hits)")
    assert hits, f"Clause direct retrieval returned nothing for {query_text!r}"
print("Clause Direct Retrieval: PASS")

## 6. Validate Clause Hybrid Retrieval

**Purpose** — Validate the full clause-side hybrid stack: BM25 + dense → normalized fusion → cross-encoder rerank, restricted to `chunk_type="clause"`.

**Explanation** — The same `metadata_filter` is threaded through **both** retrieval arms (`hybrid_search`'s 9.10 contract), so neither arm can surface a chunk the other would have excluded. The confidence gate (`evaluate_confidence`) is reported for each query.

In [ ]:
for query_text in ["What is the scope of work?",
                   "What training must the contractor provide to employer staff?",
                   "spare parts, tools and test equipment requirements"]:
    out = hybrid_pipeline(query_text, chunk_type="clause")
    hits_table(out["reranked"], "clause", top=5,
               title=f"“{query_text}” — reranked (confidence: {out['confidence']})")
    assert out["reranked"], f"Clause hybrid retrieval returned nothing for {query_text!r}"
print("Clause Hybrid Retrieval: PASS")

## 7. Validate BOQ Direct Retrieval

**Purpose** — Prove the exact BOQ-item fast path resolves real identifiers from this corpus, and that non-identifier BOQ questions fall back to BOQ-filtered dense search.

**Explanation** — `boq_direct()` mirrors `app.py`: `extract_boq_item_no()` → `get_chunk_by_boq_item_no()` (exact match on `parent` / `s_no` / `item_header_no` / `section_no`). Item `1.02.E.2` below is a real identifier present in the ingested Bill of Quantities.

In [ ]:
for query_text in ["Describe BOQ item 1.02.E.2",
                   "cooling tower supply and installation",
                   "chilled water pump starter unit"]:
    hits, path = boq_direct(query_text)
    hits_table(hits, "boq", top=6, title=f"“{query_text}” → path: `{path}` ({len(hits)} hits)")
    assert hits, f"BOQ direct retrieval returned nothing for {query_text!r}"
print("BOQ Direct Retrieval: PASS")

## 8. Validate BOQ Hybrid Retrieval

**Purpose** — Validate the identical hybrid stack over `chunk_type="boq"`, confirming BM25 lifts exact-term BOQ matches (ratings, model designations) that dense search under-ranks.

**Explanation** — Same pipeline as Section 6 with the BOQ filter; per-arm outputs are compared in Section 13.

In [ ]:
for query_text in ["2500A air circuit breaker with microprocessor release",
                   "XLPE armoured power and control cables",
                   "digital energy meter"]:
    out = hybrid_pipeline(query_text, chunk_type="boq")
    hits_table(out["reranked"], "boq", top=5,
               title=f"“{query_text}” — reranked (confidence: {out['confidence']})")
    assert out["reranked"], f"BOQ hybrid retrieval returned nothing for {query_text!r}"
print("BOQ Hybrid Retrieval: PASS")

## 9. Validate Metadata

**Purpose** — Verify per-type metadata completeness across the **entire** collection, not just sampled hits.

**Explanation** — Clause chunks must carry `clause_no`, `document_name`, `pdf_page`, `document_id`, `chapter`; BOQ chunks must carry `s_no`/`section_no` identity fields plus `unit`, `quantities`, `page_label`, `pdf_page`, `document_id`. Coverage below 100% on a *required* field is a failure; optional fields are reported for information.

In [ ]:
res = get_collection().get(include=["metadatas"])
metas = res["metadatas"]
clause_m = [m for m in metas if m.get("chunk_type") == "clause"]
boq_m    = [m for m in metas if m.get("chunk_type") == "boq"]

def coverage(ms, fields):
    return {f: sum(1 for m in ms if m.get(f) not in (None, "")) / max(len(ms), 1) for f in fields}

REQ_CLAUSE = ["clause_no", "document_name", "pdf_page", "document_id", "chapter", "heading"]
REQ_BOQ    = ["s_no", "section_no", "pdf_page", "document_id", "page_label", "schedule", "contract"]
OPT_BOQ    = ["unit", "quantities", "source_pdf", "stamp_number"]

rows = ["| Field | Type | Coverage |", "|---|---|---|"]
fails = []
for f, c in coverage(clause_m, REQ_CLAUSE).items():
    rows.append(f"| {f} | clause ({len(clause_m)}) | {c:.0%} |")
    if c < 1.0 and f != "heading": fails.append(("clause", f, c))
for f, c in coverage(boq_m, REQ_BOQ).items():
    rows.append(f"| {f} | boq ({len(boq_m)}) | {c:.0%} |")
    if c < 1.0 and f not in ("s_no",): fails.append(("boq", f, c))
for f, c in coverage(boq_m, OPT_BOQ).items():
    rows.append(f"| {f} *(optional)* | boq | {c:.0%} |")
display(Markdown("\n".join(rows)))
print("Metadata validation:", "PASS" if not fails else f"ATTENTION -> {fails}")
print("(s_no gaps are legitimate: header/continuation rows carry section_no/item_header_no instead.)")

## 10. Validate Page Retrieval

**Purpose** — Confirm every chunk resolves to an exact page under the repository's two-number rule.

**Explanation** — **Rule 1:** the *citation label* is the stamped scan number (`printed_page` for clauses, `stamp_number` for BOQ), via `prompt_engineering.get_scanned_page()`. **Rule 2:** the *file-lookup key* is `pdf_page` (with `page_number` fallback for BOQ), never the stamp — a misread stamp may only ever produce a wrong label, never a wrong image.

In [ ]:
ok_pdf = sum(1 for m in metas if resolve_pdf_page(m) is not None)
ok_stamp = sum(1 for m in metas if get_scanned_page(m) not in (None, ""))
print(f"pdf_page resolvable (Rule 2 file key): {ok_pdf}/{len(metas)}")
print(f"stamped page present (Rule 1 label)  : {ok_stamp}/{len(metas)} "
      "(unstamped cover/continuation pages are expected)")

sample_hits, _ = clause_direct("Explain clause 4.2")
h = sample_hits[0]; m = meta(h)
print(f"\nSample -- clause {clause_no(h)}: document={doc_name(h)!r}, "
      f"stamped page={scan_page(h)!r}, pdf_page={resolve_pdf_page(m)}, document_id={m.get('document_id')!r}")
assert ok_pdf == len(metas), "Every chunk must resolve a pdf_page"
print("\nPage retrieval: PASS")

## 11. Validate Page Image Retrieval

**Purpose** — Verify that `(document_id, pdf_page)` resolves to a real rendered image for retrieved chunks, exactly as `app.py`'s `AVAILABLE_PAGES` / `image_url` logic does.

**Explanation** — Whole-collection coverage is reported per document, then a clause hit and a BOQ hit are displayed with their exact page image inline.

**Fix status (Task 1):** the slug-mismatch defect identified in the pre-notebook audit has been fixed.  The fix has two parts:

1. `scripts/render_pages.py` now derives BOQ directory keys via `_slugify(pdf_filename_stem)`, identical to what `metadata_loader.py` computes for `image_document_id` — so the two always agree by construction.
2. `scripts/migrate_page_image_dirs.py` (new one-time script) renames the three existing misnamed directories on disk:
   - `…CE-10-AND-11…` → `…CE-10-CE-11…`  (3 BOQ dirs)

After running `migrate_page_image_dirs.py`, the 79 BOQ chunks with `source_pdf` (Vol-3-28-37) resolve correctly.  The 211 ADDENDUM chunks (`boq_part3.json`, no `source_pdf`) remain unresolved — see §15 Known Limitation 1.

**In this notebook** the check below reports the current on-disk state.  Run against a deployment where `migrate_page_image_dirs.py` has been applied to see the BOQ column improve from ⚠️ to ✅.

In [ ]:
from collections import Counter
cov = Counter(); miss = Counter()
for m in metas:
    (cov if page_image_path(m) else miss)[m.get("document_id")] += 1
rows = ["| document_id | chunks with image | chunks without | Note |", "|---|---|---|---|"]
for d in sorted(set(cov) | set(miss)):
    note = ""
    if "BOQ-CE-10-AND-11-LOT-4" in (d or ""):
        note = "ADDENDUM fallback id — no renders (see §15 Known Limitation 1)"
    elif "BOQ-" in (d or "") and miss.get(d, 0) > 0:
        note = "run migrate_page_image_dirs.py to fix slug mismatch"
    rows.append(f"| {d} | {cov.get(d,0)} | {miss.get(d,0)} | {note} |")
display(Markdown("\n".join(rows)))

hits, _ = clause_direct("Explain clause 1.1")
show_page_image(meta(hits[0]), caption=f"Clause {clause_no(hits[0])} — exact page")

bh, _ = boq_direct("cooling tower")
if bh:
    ok = show_page_image(meta(bh[0]), caption=f"BOQ item {boq_item(bh[0])} — exact page")
    if not ok:
        display(Markdown(
            "**BOQ image not resolved** — expected if `migrate_page_image_dirs.py` has not yet "
            "been run on this deployment. After running that script, re-start the notebook and "
            "this cell will show the rendered page."))
print("Page image retrieval validated.")

## 12. Validate Related Page Retrieval

**Purpose** — Validate both notions of "related" the system supports.

**Explanation:**

(a) **Related clauses** (unchanged): `reranker.expand_with_siblings()` pulls relevance-gated `parent_clause` siblings into the result set — their own pages/images follow automatically. This code is unchanged.

(b) **Neighboring pages (Task 2 — now shipped):** `SourceItem` in `app.py` now carries `prev_image_url` and `next_image_url`, both resolved via the same `AVAILABLE_PAGES` check that `image_url` uses.  This section validates that logic using the local `neighbor_page_images()` helper (which applies the identical algorithm) against the on-disk renders, and confirms parity with what the API would return.

In [ ]:
# (a) Sibling expansion -- related clauses (and thereby related pages)
out = hybrid_pipeline("What spare parts, tools, and test equipment must the contractor provide?",
                      chunk_type="clause", expand=True)
exp = [h for h in out["reranked"] if h.get("retrieval_source") == "sibling_expansion"]
hits_table(out["reranked"], "clause", top=8,
           title=f"Sibling expansion added {len(exp)} related clause(s)")

# (b) Neighboring pages -- validated via local helper (same logic as app.py SourceItem)
top = out["reranked"][0]
m_top = meta(top)
show_page_image(m_top, caption=f"Anchor page — clause {clause_no(top)}")

# Compute prev/next using the same algorithm app.py's _resolve_page_image_url() uses
p = resolve_pdf_page(m_top)
doc_id_top = m_top.get("document_id")
for label, delta in [("prev", -1), ("next", +1)]:
    neighbor = neighbor_page_images(m_top, radius=1)
    # Filter to the specific delta
    target_page = (p + delta) if p is not None else None
    target_path = next((path for pg, path in neighbor if pg == target_page), None)
    if target_path:
        display(Markdown(f"**{label.title()} page (pdf_page={target_page})** "
                         f"— also returned as `{label}_image_url` by `/ask` (Task 2 fix)"))
        display(IPyImage(filename=target_path, width=420))
    else:
        display(Markdown(f"*{label.title()} page (pdf_page={target_page}): no render on disk — "
                         f"`{label}_image_url` will be None in the API response*"))

print("Related-page retrieval validated:")
print("  (a) sibling expansion: repository code (unchanged)")
print("  (b) neighboring pages: now shipped as prev_image_url/next_image_url in SourceItem (Task 2)")

## 13. Compare Direct vs Hybrid

**Purpose** — Show, side by side, where each strategy wins: exact identifier queries (direct fast path is authoritative) versus free-text semantic queries (hybrid + rerank wins).

**Explanation** — For each query the top-1 of Direct and of Hybrid-reranked is compared on identifier, page, and score.

In [ ]:
COMPARE = [("Explain clause 1.2.1", "clause"),
           ("What is the scope of work?", "clause"),
           ("Describe BOQ item 1.02.E.2", "boq"),
           ("feeder for chiller motors", "boq")]
rows = ["| Query | Direct top-1 | Direct path | Hybrid top-1 | Hybrid conf. |", "|---|---|---|---|---|"]
for query_text, kind in COMPARE:
    d_hits, d_path = (clause_direct if kind == "clause" else boq_direct)(query_text)
    h_out = hybrid_pipeline(query_text, chunk_type=kind)
    ident = clause_no if kind == "clause" else boq_item
    d1 = f"{ident(d_hits[0])} (p{resolve_pdf_page(meta(d_hits[0]))})" if d_hits else "—"
    h1 = (f"{ident(h_out['reranked'][0])} (p{resolve_pdf_page(meta(h_out['reranked'][0]))})"
          if h_out["reranked"] else "—")
    rows.append(f"| {query_text} | {d1} | {d_path} | {h1} | {h_out['confidence']['top_score']} |")
display(Markdown("\n".join(rows)))
display(Markdown("**Reading:** identifier queries resolve on the exact-match fast path with score 1.0 "
                 "(hybrid never runs in production for them); free-text queries rely on fusion + reranking."))

## 14. Retrieval Evaluation Metrics

**Purpose** — Quantify retrieval quality with Hit@1 / Hit@5 / MRR over a small gold set, per strategy (dense, BM25, hybrid-fused, reranked).

**Explanation** — Gold labels: for clause queries, the expected `clause_no`; for BOQ queries, an expected phrase that must appear in the retrieved text. Small by design — this is a regression harness, not a leaderboard; extend `GOLD` as the corpus grows.

In [ ]:
GOLD = [
    # (query, chunk_type, kind of label, label)
    ("scope and purpose of this specification",       "clause", "clause_no", "1.1"),
    ("verification and validation of design",         "clause", "clause_no", "3.2"),
    ("training of employer staff",                    "clause", "text",      "training"),
    ("operation and maintenance manuals",             "clause", "text",      "manual"),
    ("2500A TPN air circuit breaker",                 "boq",    "text",      "2500"),
    ("star delta starter for chilled water pump",     "boq",    "text",      "star"),
    ("current transformer for metering",              "boq",    "text",      "metering"),
    ("XLPE armoured cable termination",               "boq",    "text",      "xlpe"),
]
def is_match(hit, kind, label):
    if kind == "clause_no":
        return meta(hit).get("clause_no") == label
    return label.lower() in (hit.get("document") or "").lower()

def rank_of(hits, kind, label):
    for i, h in enumerate(hits, 1):
        if is_match(h, kind, label): return i
    return None

import statistics
strategies = ["dense", "bm25", "hybrid", "reranked"]
scores = {s: [] for s in strategies}
for query_text, ct, kind, label in GOLD:
    out = hybrid_pipeline(query_text, chunk_type=ct, top_n=10)
    for s in strategies:
        scores[s].append(rank_of(out[s], kind, label))

rows = ["| Strategy | Hit@1 | Hit@5 | MRR |", "|---|---|---|---|"]
for s in strategies:
    rk = scores[s]
    hit1 = sum(1 for r in rk if r == 1) / len(rk)
    hit5 = sum(1 for r in rk if r and r <= 5) / len(rk)
    mrr  = sum((1 / r) if r else 0 for r in rk) / len(rk)
    rows.append(f"| {s} | {hit1:.2f} | {hit5:.2f} | {mrr:.2f} |")
display(Markdown("\n".join(rows)))
METRIC_RANKS = scores  # used by failure analysis below

## 15. Failure Analysis

**Purpose** — Surface every gold-set query any strategy missed, diagnose it, and record the audit's structural gaps as concrete improvement recommendations.

**Explanation** — A "failure" is a gold label absent from a strategy's top-10. Diagnoses distinguish *vocabulary mismatch* (dense misses exact terms), *lexical-only phrasing* (BM25 misses paraphrase), and *corpus absence* (the concept genuinely isn't in the 353-chunk corpus — the correct behavior is the low-confidence gate, not a forced answer).

In [ ]:
rows = ["| Query | dense | bm25 | hybrid | reranked | Diagnosis |", "|---|---|---|---|---|---|"]
any_fail = False
for i, (query_text, ct, kind, label) in enumerate(GOLD):
    rk = {s: METRIC_RANKS[s][i] for s in METRIC_RANKS}
    if all(v == 1 for v in rk.values()):
        continue
    any_fail = True
    if rk["reranked"] is None and rk["hybrid"] is None:
        diag = "not retrieved by any arm — check corpus coverage / label"
    elif rk["dense"] in (None,) and rk["bm25"]:
        diag = "vocabulary mismatch — BM25 rescued it (hybrid design working)"
    elif rk["bm25"] in (None,) and rk["dense"]:
        diag = "paraphrase query — dense rescued it (hybrid design working)"
    elif rk["reranked"] and rk["reranked"] < min(x for x in (rk["dense"], rk["bm25"]) if x):
        diag = "reranker promoted the right chunk (expected behavior)"
    else:
        diag = "rank drift — candidate present, ordering imperfect"
    rows.append(f"| {query_text} | {rk['dense']} | {rk['bm25']} | {rk['hybrid']} | {rk['reranked']} | {diag} |")
display(Markdown("\n".join(rows) if any_fail else "**All gold queries hit at rank 1 across all strategies.**"))

### Findings & Improvements Status (post-fix audit)

The following issues were identified in the pre-fix audit. Status reflects the `dmrc_deploy_fixes.zip` changes.

| # | Finding | Status |
|---|---|---|
| 1 | **Neighboring-page image retrieval** missing from repository | ✅ **FIXED** — `prev_image_url` / `next_image_url` added to `SourceItem` in `app.py` (Task 2) |
| 2 | **Figure retrieval dormant** — `figure_images/manifest.json` not committed | ✅ **DOCUMENTED** — confirmed 0 extractable figures from scanned PDFs; feature activates automatically when embedded-diagram documents are ingested |
| 3 | **BOQ page images 0/290** — slug mismatch (`…CE-10-CE-11…` vs `…CE-10-AND-11…`) | ✅ **FIXED** — `render_pages.py` now derives keys via `_slugify(filename)`; `migrate_page_image_dirs.py` renames existing dirs (Task 1) |
| 4 | **ADDENDUM chunks (211) can't resolve images** — no `source_pdf`, unknown PDF-page offset | ⚠️ **NOT FIXABLE AUTOMATICALLY** — requires manual offset verification; documented in CHANGELOG Known Limitation 1 |
| 5 | **Clause/BOQ share one unfiltered retrieval pool** for free-text queries | ✅ **FIXED** — `src/query_router.py` classifies intent and passes `chunk_type` filter to `hybrid_search()` (Task 4) |
| 6 | **Fusion is max-merge, not RRF** | 📌 Documented — adequate at current corpus size |
| 7 | **Confidence thresholds uncalibrated** | 📌 `scripts/calibrate_confidence.py` exists; re-run after corpus changes |

**Remaining action items:**
- Run `python scripts/migrate_page_image_dirs.py` on every deployment that has pre-rendered page_images
- Re-render `page_images/` with updated `render_pages.py` on new deployments (keys now correct)
- Resolve ADDENDUM pdf_page offset manually to unlock the remaining 211 BOQ image links